# RQ-4 — Wirken sich Metadaten auf die Visualisierungsqualität aus?

Dieses Notebook wertet die Ablationsstudie zu Forschungsfrage 4 aus. Verglichen werden Agenten-Läufe, in denen der Metadaten-Schritt aktiviert war, mit Läufen, in denen er deaktiviert war. Beide Bedingungen liegen für die Konfigurationen *DE+Python* und *EN+Python* vor.

Die Auswertung deckt folgende Punkte ab:

- paarweiser Vergleich der SEVQ-Werte (Total und sechs Einzeldimensionen),
- paarweiser Vergleich der VER-Erfolgsraten,
- Berechnung von ΔSEVQ und der Metadata Benefit Rate (MBR),
- Auswertung der A/B-Präferenzfragen aus der Online-Studie (Fragen 25–30),
- gepaarte statistische Tests mit Bonferroni-Korrektur,
- Erzeugung der Abbildungen für die Thesis und ein JSON-Export der Kennzahlen.

Das Notebook gehört zum Reproduktionspaket der Masterarbeit. Wer die Auswertung nachvollziehen möchte, führt die Zellen einfach von oben nach unten aus.


## Setup

Erst die Bibliotheken laden. `matplotlib` läuft im `Agg`-Backend, damit das Notebook auch in einer Headless-Umgebung (CI, Server) ohne Display funktioniert. `seaborn` übernimmt das Theme für die Plots, `scipy.stats` liefert die Tests.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json

from scipy import stats as sp_stats

## Konfiguration und Pfade

Hier sind alle festen Werte gesammelt, die im Rest des Notebooks benutzt werden:

- `METRICS` — die sechs SEVQ-Dimensionen.
- `MODEL_MAPPING` — interne Modellnamen → Anzeigenamen für die Plots.
- `BASE_PATH` — Wurzelverzeichnis der Agenten-Outputs (50 Datensätze, je Konfiguration ein Unterordner).
- `STUDY_PATH` — CSV-Export der Online-Studie aus LamaPoll.
- `OUTPUT_DIR` / `THESIS_FIG_DIR` — Zielordner für die generierten Abbildungen. Die Plots werden zusätzlich direkt in den Thesis-Quellbaum geschrieben, damit die Typst-Datei sie ohne Umweg einbinden kann.
- `CONFIGS` — die zwei Bedingungen, die für RQ-4 verglichen werden.
- `AB_QUESTIONS` — Zuordnung von Studienfragen zu Konfigurationen für den A/B-Teil.
- `ALPHA_CORR` — Bonferroni-korrigiertes Signifikanzniveau (zwei Hauptvergleiche pro Konfiguration: SEVQ und VER).


In [ ]:
METRICS = ['bugs', 'transformation', 'compliance', 'type', 'encoding', 'aesthetics']
MODEL_MAPPING = {
    'gpt-5': 'GPT-5',
    'gpt-4o': 'GPT-4o',
    'google/gemini-3-flash-preview': 'Gemini',
    'x-ai/grok-4.1-fast': 'Grok',
    'anthropic/claude-sonnet-4.5': 'Claude Sonnet'
}

BASE_PATH = Path("./src/resources/paper_results/50_dataset_run")
STUDY_PATH = Path("./results/rq_4/study_data.csv")
OUTPUT_DIR = Path("./results/rq_4/figures")
THESIS_FIG_DIR = OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
THESIS_FIG_DIR.mkdir(parents=True, exist_ok=True)

CONFIGS = {
    'DE+Python': {
        'meta': BASE_PATH / 'DE' / 'Python' / 'output',
        'no_meta': BASE_PATH / 'no_metadata' / 'DE' / 'Python' / 'output',
    },
    'EN+Python': {
        'meta': BASE_PATH / 'EN' / 'Python' / 'output',
        'no_meta': BASE_PATH / 'no_metadata' / 'EN' / 'Python' / 'output',
    }
}

AB_QUESTIONS = {
    'DE+Python': [25, 26, 27],
    'EN+Python': [28, 29, 30],
}

ALPHA = 0.05
N_TESTS_FAMILY = 2
ALPHA_CORR = ALPHA / N_TESTS_FAMILY

sns.set_theme(style='whitegrid', context='talk', font='DejaVu Sans')
PALETTE_META = {'Mit Metadaten': '#009B91', 'Ohne Metadaten': '#334152'}

## Daten laden

Die Loader-Funktionen lesen die Rohdaten direkt aus den Output-Verzeichnissen der Agenten-Läufe. Pro Datensatz liegt jeweils eine `sevq.csv` (mehrere Bewertungen je Visualisierung) und eine `VER_0.csv` (Visualization Error Rate, ein Eintrag pro Visualisierung) vor. Beide werden über alle Datensätze hinweg zu einem langen DataFrame zusammengefügt; die `dataset_id` bleibt als Spalte erhalten, damit später paarweise gemerged werden kann.

`load_study_data` lädt zusätzlich die exportierte Studientabelle. LamaPoll exportiert mit Semikolon als Trenner — daher der explizite `sep=';'`.


In [ ]:
def load_sevq(output_dir: Path) -> pd.DataFrame:
    rows = []
    for dataset_dir in sorted(output_dir.iterdir()):
        if not dataset_dir.is_dir():
            continue
        sevq_file = dataset_dir / 'sevq.csv'
        if not sevq_file.exists():
            continue
        df = pd.read_csv(sevq_file)
        df['dataset_id'] = dataset_dir.name
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def load_ver(output_dir: Path) -> pd.DataFrame:
    rows = []
    for dataset_dir in sorted(output_dir.iterdir()):
        if not dataset_dir.is_dir():
            continue
        ver_file = dataset_dir / 'VER_0.csv'
        if not ver_file.exists():
            continue
        df = pd.read_csv(ver_file)
        df['dataset_id'] = dataset_dir.name
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def load_study_data() -> pd.DataFrame:
    return pd.read_csv(STUDY_PATH, sep=';', encoding='utf-8')

## Aggregation pro Datensatz

Die Roh-CSVs enthalten je nach Datensatz mehrere Bewertungen. Für die paarweisen Tests wird pro Datensatz genau ein Wert benötigt:

- `compute_sevq_total_per_dataset` — summiert die sechs SEVQ-Dimensionen je Zeile und mittelt anschließend über alle Bewertungen eines Datensatzes (Wertebereich 0–60).
- `compute_sevq_dimensions_per_dataset` — gleicher Schritt, aber pro Dimension einzeln; wird für die Heatmap und die Dimensionstests gebraucht.
- `compute_ver_per_dataset` — bildet die Erfolgsrate als Anteil erfolgreicher Visualisierungen pro Datensatz in Prozent.


In [ ]:
def compute_sevq_total_per_dataset(sevq_df: pd.DataFrame) -> pd.DataFrame:
    if sevq_df.empty:
        return pd.DataFrame(columns=['dataset_id', 'sevq_total'])
    sevq_df = sevq_df.copy()
    sevq_df['total'] = sevq_df[METRICS].sum(axis=1)
    result = sevq_df.groupby('dataset_id')['total'].mean().reset_index()
    result.columns = ['dataset_id', 'sevq_total']
    return result


def compute_sevq_dimensions_per_dataset(sevq_df: pd.DataFrame) -> pd.DataFrame:
    if sevq_df.empty:
        return pd.DataFrame()
    return sevq_df.groupby('dataset_id')[METRICS].mean().reset_index()


def compute_ver_per_dataset(ver_df: pd.DataFrame) -> pd.DataFrame:
    if ver_df.empty:
        return pd.DataFrame(columns=['dataset_id', 'ver_success_rate'])
    result = ver_df.groupby('dataset_id').agg(
        total_vis=('ver_value', 'count'),
        successful_vis=('ver_value', 'sum')
    ).reset_index()
    result['ver_success_rate'] = (result['successful_vis'] / result['total_vis']) * 100
    return result[['dataset_id', 'ver_success_rate']]

## Statistische Hilfsfunktionen

Kleine, wiederverwendbare Bausteine für die Auswertung. Alles ist absichtlich kurz gehalten, damit der eigentliche Analyseschritt weiter unten lesbar bleibt.

- `ci95` — 95%-Konfidenzintervall für den Mittelwert über die t-Verteilung.
- `paired_ttest` — gepaarter t-Test; gibt zusätzlich Cohen's *d* und das KI der Mittelwert-Differenz zurück.
- `wilcoxon_test` — nicht-parametrische Alternative für den Fall, dass die Differenzen nicht normalverteilt sind. Effektstärke *r* wird aus dem p-Wert geschätzt.
- `wilson_ci` — Wilson-Score-KI für Anteilswerte (numerisch stabiler als das Normal-Approximations-KI, gerade bei kleinen n).
- `binomial_test_ab` — zweiseitiger Binomialtest gegen 50% für die A/B-Präferenzfragen.
- `metadata_benefit_rate` — fasst eine Serie paarweiser Differenzen zur *Metadata Benefit Rate* (MBR) zusammen, also dem Anteil der Datensätze, in denen Metadaten den Score verbessert haben.


In [ ]:
def ci95(vals):
    """95% CI for the mean via t-distribution."""
    vals = np.asarray(vals)
    n = len(vals)
    se = sp_stats.sem(vals)
    t_crit = sp_stats.t.ppf(0.975, n - 1)
    return (vals.mean() - t_crit * se, vals.mean() + t_crit * se)


def paired_ttest(x, y):
    """Paired t-test returning t, p, Cohen's d, and CI of mean difference."""
    diff = np.asarray(x) - np.asarray(y)
    n = len(diff)
    t_stat, p_val = sp_stats.ttest_rel(x, y)
    d = float(diff.mean() / diff.std(ddof=1)) if diff.std(ddof=1) > 0 else 0.0
    se = sp_stats.sem(diff)
    t_crit = sp_stats.t.ppf(0.975, n - 1)
    ci_lo = float(diff.mean() - t_crit * se)
    ci_hi = float(diff.mean() + t_crit * se)
    return {'t': float(t_stat), 'p': float(p_val), 'd': d, 'n': n,
            'ci_lower': ci_lo, 'ci_upper': ci_hi}


def wilcoxon_test(x, y):
    """Wilcoxon signed-rank test."""
    diff = np.asarray(x) - np.asarray(y)
    diff_nz = diff[diff != 0]
    if len(diff_nz) < 2:
        return {'W': np.nan, 'p': 1.0, 'r': 0.0, 'n': len(x)}
    w_stat, p_val = sp_stats.wilcoxon(diff_nz)
    r = abs(sp_stats.norm.ppf(p_val / 2)) / np.sqrt(len(diff_nz))
    return {'W': float(w_stat), 'p': float(p_val), 'r': float(r), 'n': len(x)}


def wilson_ci(p_hat, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    denom = 1 + z ** 2 / n
    center = (p_hat + z ** 2 / (2 * n)) / denom
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z ** 2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)


def binomial_test_ab(n_meta, n_total):
    """Two-sided binomial test against 50%."""
    result = sp_stats.binomtest(n_meta, n_total, p=0.5, alternative='two-sided')
    p_hat = n_meta / n_total
    ci_low, ci_high = wilson_ci(p_hat, n_total)
    return {
        'p': float(result.pvalue),
        'prop_meta': p_hat,
        'ci_lower': ci_low,
        'ci_upper': ci_high,
        'n_meta': n_meta,
        'n_no_meta': n_total - n_meta,
        'n_total': n_total
    }


def metadata_benefit_rate(delta_series):
    """Compute MBR and breakdown counts for a series of paired differences."""
    n_improved = int((delta_series > 0).sum())
    n_equal = int((delta_series == 0).sum())
    n_worse = int((delta_series < 0).sum())
    mbr = n_improved / len(delta_series) * 100
    return {
        'mean': float(delta_series.mean()),
        'sd': float(delta_series.std()),
        'median': float(delta_series.median()),
        'mbr': mbr,
        'n_improved': n_improved,
        'n_equal': n_equal,
        'n_worse': n_worse,
    }

## Statistische Analyse

Die ursprüngliche Auswertung steckte komplett in einer einzigen großen Funktion. Für die Reproduzierbarkeit ist es schöner, die Schritte zu trennen — so kann jede Phase einzeln gelesen, getestet oder geändert werden, ohne dass der Rest mitleidet.

Die Analyse besteht aus vier Bausteinen:

1. eine einzelne Konfiguration auswerten,
2. die Ergebnisse über beide Konfigurationen aggregieren,
3. die A/B-Präferenzfragen aus der Studie auswerten,
4. eine schlanke Orchestrierungsfunktion, die alles zusammenhält.

### Schritt 1 — eine einzelne Konfiguration auswerten

`analyze_config` arbeitet folgenden Ablauf für *eine* Konfiguration ab (z.B. `DE+Python`):

1. Rohdaten der beiden Bedingungen (mit / ohne Metadaten) laden.
2. Pro Datensatz aggregieren (SEVQ-Total, SEVQ-Dimensionen, VER).
3. Schnittmenge der Datensatz-IDs bilden — nur Datensätze, die in *beiden* Bedingungen vorliegen, gehen in die paarweisen Tests ein.
4. Deskriptive Kennzahlen ausgeben (M, SD, Median, 95%-KI).
5. Inferenzstatistik rechnen: gepaarter t-Test und Wilcoxon-Test für SEVQ-Total und VER.
6. ΔSEVQ und MBR berechnen.
7. Die sechs SEVQ-Dimensionen einzeln testen.
8. ΔVER und die zugehörige MBR ergänzen.

Das zurückgegebene Dictionary enthält alle Zwischenergebnisse, die später für Plots und Export gebraucht werden.


In [ ]:
def analyze_config(config_name, paths):
    """Wertet eine einzelne Konfiguration (z.B. DE+Python) aus.

    Schritte: Rohdaten laden -> pro Datensatz aggregieren ->
    Schnittmenge bilden -> deskriptive und inferenzstatistische
    Auswertung -> Dictionary mit allen Ergebnissen zurückgeben.
    """
    print(f"\n{'='*50}")
    print(f"Konfiguration: {config_name}")
    print(f"{'='*50}")

    # 1) Rohdaten beider Bedingungen laden
    sevq_meta = load_sevq(paths['meta'])
    sevq_no_meta = load_sevq(paths['no_meta'])
    ver_meta = load_ver(paths['meta'])
    ver_no_meta = load_ver(paths['no_meta'])

    # 2) Pro Datensatz aggregieren
    sevq_total_meta = compute_sevq_total_per_dataset(sevq_meta)
    sevq_total_no_meta = compute_sevq_total_per_dataset(sevq_no_meta)
    sevq_dims_meta = compute_sevq_dimensions_per_dataset(sevq_meta)
    sevq_dims_no_meta = compute_sevq_dimensions_per_dataset(sevq_no_meta)
    ver_meta_ds = compute_ver_per_dataset(ver_meta)
    ver_no_meta_ds = compute_ver_per_dataset(ver_no_meta)

    print(f"\nDatensätze mit SEVQ (meta): {len(sevq_total_meta)}")
    print(f"Datensätze mit SEVQ (no_meta): {len(sevq_total_no_meta)}")
    print(f"Datensätze mit VER (meta): {len(ver_meta_ds)}")
    print(f"Datensätze mit VER (no_meta): {len(ver_no_meta_ds)}")

    # 3) Nur Datensätze, die in beiden Bedingungen vorliegen, werden gepaart
    common_sevq = set(sevq_total_meta['dataset_id']) & set(sevq_total_no_meta['dataset_id'])
    common_ver = set(ver_meta_ds['dataset_id']) & set(ver_no_meta_ds['dataset_id'])
    print(f"Gemeinsame Datensätze (SEVQ): {len(common_sevq)}")
    print(f"Gemeinsame Datensätze (VER): {len(common_ver)}")

    sevq_paired = pd.merge(
        sevq_total_meta[sevq_total_meta['dataset_id'].isin(common_sevq)],
        sevq_total_no_meta[sevq_total_no_meta['dataset_id'].isin(common_sevq)],
        on='dataset_id', suffixes=('_meta', '_no_meta')
    ).sort_values('dataset_id')

    sevq_dims_paired = pd.merge(
        sevq_dims_meta[sevq_dims_meta['dataset_id'].isin(common_sevq)],
        sevq_dims_no_meta[sevq_dims_no_meta['dataset_id'].isin(common_sevq)],
        on='dataset_id', suffixes=('_meta', '_no_meta')
    ).sort_values('dataset_id')

    ver_paired = pd.merge(
        ver_meta_ds[ver_meta_ds['dataset_id'].isin(common_ver)],
        ver_no_meta_ds[ver_no_meta_ds['dataset_id'].isin(common_ver)],
        on='dataset_id', suffixes=('_meta', '_no_meta')
    ).sort_values('dataset_id')

    # 4) Deskriptive Statistik
    print(f"\n--- SEVQ Total ---")
    for cond, col in [('Meta', 'sevq_total_meta'), ('NoMeta', 'sevq_total_no_meta')]:
        vals = sevq_paired[col]
        lo, hi = ci95(vals)
        print(f"  {cond}: M={vals.mean():.2f}, SD={vals.std():.2f}, "
              f"Median={vals.median():.2f}, CI95=[{lo:.2f}; {hi:.2f}]")

    print(f"\n--- VER Success Rate ---")
    for cond, col in [('Meta', 'ver_success_rate_meta'), ('NoMeta', 'ver_success_rate_no_meta')]:
        vals = ver_paired[col]
        lo, hi = ci95(vals)
        print(f"  {cond}: M={vals.mean():.1f}%, SD={vals.std():.1f}%, "
              f"Median={vals.median():.1f}%, CI95=[{lo:.1f}; {hi:.1f}]")

    # 5) Inferenzstatistik (gepaart, t-Test + Wilcoxon)
    print(f"\n--- Statistical Tests (alpha_corr = {ALPHA_CORR:.4f}) ---")

    sevq_test = paired_ttest(
        sevq_paired['sevq_total_meta'].values,
        sevq_paired['sevq_total_no_meta'].values
    )
    print(f"  SEVQ t-test: t={sevq_test['t']:.3f}, p={sevq_test['p']:.4f}, "
          f"d={sevq_test['d']:.3f}, CI_diff=[{sevq_test['ci_lower']:.2f}; {sevq_test['ci_upper']:.2f}]")

    sevq_wilcox = wilcoxon_test(
        sevq_paired['sevq_total_meta'].values,
        sevq_paired['sevq_total_no_meta'].values
    )
    print(f"  SEVQ Wilcoxon: W={sevq_wilcox['W']:.1f}, p={sevq_wilcox['p']:.4f}, r={sevq_wilcox['r']:.3f}")

    ver_test = paired_ttest(
        ver_paired['ver_success_rate_meta'].values,
        ver_paired['ver_success_rate_no_meta'].values
    )
    print(f"  VER t-test: t={ver_test['t']:.3f}, p={ver_test['p']:.4f}, "
          f"d={ver_test['d']:.3f}, CI_diff=[{ver_test['ci_lower']:.2f}; {ver_test['ci_upper']:.2f}]")

    ver_wilcox = wilcoxon_test(
        ver_paired['ver_success_rate_meta'].values,
        ver_paired['ver_success_rate_no_meta'].values
    )
    print(f"  VER Wilcoxon: W={ver_wilcox['W']:.1f}, p={ver_wilcox['p']:.4f}, r={ver_wilcox['r']:.3f}")

    # 6) ΔSEVQ und Metadata Benefit Rate
    sevq_paired['delta_sevq'] = sevq_paired['sevq_total_meta'] - sevq_paired['sevq_total_no_meta']
    delta_stats = metadata_benefit_rate(sevq_paired['delta_sevq'])
    print(f"\n--- ΔSEVQ (Meta - NoMeta) ---")
    print(f"  M={delta_stats['mean']:.2f}, SD={delta_stats['sd']:.2f}, "
          f"Median={delta_stats['median']:.2f}")
    print(f"  MBR: {delta_stats['mbr']:.1f}% ({delta_stats['n_improved']} besser, "
          f"{delta_stats['n_equal']} gleich, {delta_stats['n_worse']} schlechter)")

    # 7) SEVQ-Dimensionen einzeln testen
    print(f"\n--- SEVQ Dimensions (Meta vs. NoMeta) ---")
    dim_results = {}
    for dim in METRICS:
        meta_vals = sevq_dims_paired[f'{dim}_meta'].values
        no_meta_vals = sevq_dims_paired[f'{dim}_no_meta'].values
        diff = meta_vals - no_meta_vals
        t_result = paired_ttest(meta_vals, no_meta_vals)
        dim_results[dim] = {
            'meta_mean': float(meta_vals.mean()),
            'no_meta_mean': float(no_meta_vals.mean()),
            'diff_mean': float(diff.mean()),
            't': t_result['t'], 'p': t_result['p'], 'd': t_result['d']
        }
        print(f"  {dim}: Meta={meta_vals.mean():.2f}, NoMeta={no_meta_vals.mean():.2f}, "
              f"Δ={diff.mean():+.2f}, t={t_result['t']:.2f}, p={t_result['p']:.4f}, d={t_result['d']:.3f}")

    # 8) ΔVER und MBR(VER)
    ver_paired['delta_ver'] = ver_paired['ver_success_rate_meta'] - ver_paired['ver_success_rate_no_meta']
    delta_ver_stats = metadata_benefit_rate(ver_paired['delta_ver'])
    print(f"\n--- ΔVER (Meta - NoMeta) ---")
    print(f"  M={delta_ver_stats['mean']:.1f}%, SD={delta_ver_stats['sd']:.1f}%")
    print(f"  MBR(VER): {delta_ver_stats['mbr']:.1f}%")

    return {
        'sevq_paired': sevq_paired,
        'ver_paired': ver_paired,
        'sevq_dims_paired': sevq_dims_paired,
        'sevq_test': sevq_test,
        'ver_test': ver_test,
        'sevq_wilcox': sevq_wilcox,
        'ver_wilcox': ver_wilcox,
        'dim_results': dim_results,
        'delta_stats': delta_stats,
        'delta_ver_stats': delta_ver_stats,
        'n_meta_sevq': len(sevq_total_meta),
        'n_no_meta_sevq': len(sevq_total_no_meta),
        'n_meta_ver': len(ver_meta_ds),
        'n_no_meta_ver': len(ver_no_meta_ds),
        'sevq_total_meta': sevq_total_meta,
        'sevq_total_no_meta': sevq_total_no_meta,
    }


### Schritt 2 — Aggregation über beide Konfigurationen

Nachdem jede Konfiguration einzeln ausgewertet ist, wird ein gemeinsamer Blick auf alle paarweisen Differenzen geworfen. Das ist die Kennzahl, mit der in der Thesis argumentiert wird, ob Metadaten *insgesamt* helfen — unabhängig von Sprache.


In [ ]:
def print_global_aggregate(all_results):
    """Aggregierte ΔSEVQ- und ΔVER-Statistiken über alle Konfigurationen."""
    print(f"\n{'='*50}")
    print("AGGREGIERT (beide Konfigurationen)")
    print(f"{'='*50}")

    all_delta_sevq = pd.concat([r['sevq_paired']['delta_sevq'] for r in all_results.values()])
    all_delta_ver = pd.concat([r['ver_paired']['delta_ver'] for r in all_results.values()])
    print(f"  ΔSEVQ global: M={all_delta_sevq.mean():.2f}, SD={all_delta_sevq.std():.2f}")
    print(f"  MBR(SEVQ) global: {(all_delta_sevq > 0).sum() / len(all_delta_sevq) * 100:.1f}%")
    print(f"  ΔVER global: M={all_delta_ver.mean():.1f}%, SD={all_delta_ver.std():.1f}%")
    print(f"  MBR(VER) global: {(all_delta_ver > 0).sum() / len(all_delta_ver) * 100:.1f}%")


### Schritt 3 — A/B-Präferenzen aus der Online-Studie

Die Fragen 25–30 der Online-Studie sind paarweise A/B-Vergleiche zwischen einer Visualisierung *mit* und einer *ohne* Metadaten. Variante 1 entspricht der Metadaten-Bedingung, Variante 2 der Bedingung ohne Metadaten.

Pro Konfiguration werden die Stimmen über die jeweils drei zugehörigen Fragen aufsummiert und ein zweiseitiger Binomialtest gegen 50% gerechnet. Anschließend wird derselbe Test noch einmal über alle sechs Fragen zusammen ausgeführt — das ist der Wert, der in der Thesis als „Gesamt“ auftaucht.

Die `replace`-Aufrufe entfernen unsichtbare Zero-Width-Zeichen, die LamaPoll beim CSV-Export gelegentlich in die Antwort-Strings einbettet. Ohne die Bereinigung würden die `startswith`-Vergleiche stillschweigend fehlschlagen.


In [ ]:
def analyze_ab_study():
    """Wertet die A/B-Präferenzfragen (25-30) aus den Studiendaten aus."""
    print(f"\n{'='*50}")
    print("A/B-VERGLEICHE (Studiendaten)")
    print(f"{'='*50}")

    study_df = load_study_data()
    ab_results = {}

    for config_name, questions in AB_QUESTIONS.items():
        print(f"\n--- {config_name} (Fragen {questions}) ---")
        n_meta_total = 0
        n_total = 0

        for q_num in questions:
            col_name = f'Frage {q_num} - Wählen Sie die Ihrer Meinung nach passendere Variante aus'
            if col_name not in study_df.columns:
                print(f"  Warnung: Spalte '{col_name}' nicht gefunden!")
                continue
            responses = study_df[col_name].dropna()
            # Unsichtbare Zero-Width-Zeichen aus dem LamaPoll-Export entfernen
            responses = responses.str.replace('\u200b', '').str.replace('\u200c', '').str.strip()
            n_v1 = responses.str.startswith('Variante 1').sum()
            n_v2 = responses.str.startswith('Variante 2').sum()
            n_q = n_v1 + n_v2
            print(f"  Frage {q_num}: V1(Meta)={n_v1}, V2(NoMeta)={n_v2}, n={n_q}")
            n_meta_total += n_v1
            n_total += n_q

        binom = binomial_test_ab(n_meta_total, n_total)
        print(f"  GESAMT: Meta={n_meta_total} ({binom['prop_meta']*100:.1f}%), "
              f"NoMeta={n_total-n_meta_total} ({(1-binom['prop_meta'])*100:.1f}%), n={n_total}")
        print(f"  Binomialtest: p={binom['p']:.4f}, "
              f"CI95=[{binom['ci_lower']*100:.1f}%; {binom['ci_upper']*100:.1f}%]")
        ab_results[config_name] = binom

    # Gesamt über beide Konfigurationen
    total_meta = sum(r['n_meta'] for r in ab_results.values())
    total_n = sum(r['n_total'] for r in ab_results.values())
    overall_binom = binomial_test_ab(total_meta, total_n)
    print(f"\n  OVERALL: Meta={total_meta} ({overall_binom['prop_meta']*100:.1f}%), "
          f"NoMeta={total_n-total_meta} ({(1-overall_binom['prop_meta'])*100:.1f}%), n={total_n}")
    print(f"  Binomialtest: p={overall_binom['p']:.4f}")
    ab_results['Gesamt'] = overall_binom

    return ab_results


### Schritt 4 — Orchestrierung

`run_analysis` ist nach dem Aufteilen nur noch eine kurze Klammer um die einzelnen Schritte. Diese Funktion ist gleichzeitig der Einstiegspunkt, der unten in der letzten Zelle aufgerufen wird.


In [ ]:
def run_analysis():
    """Führt die komplette RQ-4-Auswertung aus."""
    print("=" * 70)
    print("RQ-4 ANALYSIS: Einfluss von Metadaten")
    print("=" * 70)

    all_results = {}
    for config_name, paths in CONFIGS.items():
        all_results[config_name] = analyze_config(config_name, paths)

    print_global_aggregate(all_results)
    ab_results = analyze_ab_study()

    generate_plots(all_results, ab_results)
    export_summary(all_results, ab_results)

    print(f"\n{'='*70}")
    print("Analysis complete. Figures saved to:", OUTPUT_DIR)
    print(f"{'='*70}")


## Visualisierungen für die Thesis

In den folgenden Zellen sind die einzelnen Plot-Funktionen definiert. Jede Funktion erzeugt eine Abbildung, die in der Thesis wiederverwendet wird. Die Plots werden über den Helfer `_save_fig` jeweils als SVG (für Typst) und als PNG (für die Vorschau) exportiert — und gleichzeitig in `OUTPUT_DIR` und in den Thesis-Quellbaum geschrieben.


In [ ]:
def _save_fig(fig, name):
    """Save figure to both output and thesis directories."""
    for d in (OUTPUT_DIR, THESIS_FIG_DIR):
        fig.savefig(d / f'{name}.svg', bbox_inches='tight', format='svg')
        fig.savefig(d / f'{name}.png', bbox_inches='tight', dpi=200)
    plt.close(fig)
    print(f"  -> {name}.svg")

### SEVQ-Total — Boxplot pro Konfiguration

Boxplot-Vergleich der SEVQ-Total-Werte (mit vs. ohne Metadaten), getrennt nach Konfiguration. Auf die Box wird der Mittelwert als schwarze Raute eingezeichnet, darüber stehen *M* und *n*. Die Signifikanz-Klammer zeigt p-Wert und Cohen's *d* aus dem gepaarten t-Test.


In [ ]:
def plot_sevq_boxplot(all_results):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    for idx, (config_name, res) in enumerate(all_results.items()):
        ax = axes[idx]
        data = []
        for _, row in res['sevq_total_meta'].iterrows():
            data.append({'dataset_id': row['dataset_id'], 'Bedingung': 'Mit Metadaten',
                         'SEVQ Total': row['sevq_total']})
        for _, row in res['sevq_total_no_meta'].iterrows():
            data.append({'dataset_id': row['dataset_id'], 'Bedingung': 'Ohne Metadaten',
                         'SEVQ Total': row['sevq_total']})
        plot_df = pd.DataFrame(data)

        sns.boxplot(data=plot_df, x='Bedingung', y='SEVQ Total',
                    palette=PALETTE_META, ax=ax, width=0.5,
                    showmeans=True, meanprops=dict(marker='D', markerfacecolor='black', markersize=8))
        sns.stripplot(data=plot_df, x='Bedingung', y='SEVQ Total',
                      color='black', alpha=0.3, size=4, jitter=True, ax=ax)

        for i, cond in enumerate(['Mit Metadaten', 'Ohne Metadaten']):
            vals = plot_df[plot_df['Bedingung'] == cond]['SEVQ Total']
            ax.text(i, vals.max() + 0.5, f'$M={vals.mean():.1f}$\n$n={len(vals)}$',
                    ha='center', va='bottom', fontsize=11)

        test = res['sevq_test']
        sig_text = f"$p={test['p']:.3f}$" if test['p'] >= 0.001 else "$p<0.001$"
        sig_text += f"\n$d={test['d']:.2f}$"
        y_max = plot_df['SEVQ Total'].max() + 3
        ax.plot([0, 0, 1, 1], [y_max, y_max + 0.3, y_max + 0.3, y_max], 'k-', linewidth=1)
        ax.text(0.5, y_max + 0.5, sig_text, ha='center', va='bottom', fontsize=10)

        ax.set_title(config_name, fontsize=14, fontweight='bold')
        ax.set_xlabel('')
        if idx == 0:
            ax.set_ylabel('SEVQ Total (0–60)')
        ax.set_ylim(20, 62)

    fig.suptitle('SEVQ-Total: Mit vs. Ohne Metadaten', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    _save_fig(fig, '05-rq4-sevq-boxplot')

### VER — Boxplot pro Konfiguration

Gleiche Darstellung wie der SEVQ-Boxplot, nur für die VER-Erfolgsrate (in Prozent). Die y-Achse ist großzügig skaliert, damit auch Datensätze mit 0%- oder 100%-Werten sauber dargestellt werden.


In [ ]:
def plot_ver_boxplot(all_results):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    for idx, (config_name, res) in enumerate(all_results.items()):
        ax = axes[idx]
        paired = res['ver_paired']
        data = []
        for _, row in paired.iterrows():
            data.append({'Bedingung': 'Mit Metadaten', 'VER (%)': row['ver_success_rate_meta']})
            data.append({'Bedingung': 'Ohne Metadaten', 'VER (%)': row['ver_success_rate_no_meta']})
        plot_df = pd.DataFrame(data)

        sns.boxplot(data=plot_df, x='Bedingung', y='VER (%)',
                    palette=PALETTE_META, ax=ax, width=0.5,
                    showmeans=True, meanprops=dict(marker='D', markerfacecolor='black', markersize=8))
        sns.stripplot(data=plot_df, x='Bedingung', y='VER (%)',
                      color='black', alpha=0.3, size=4, jitter=True, ax=ax)

        for i, cond in enumerate(['Mit Metadaten', 'Ohne Metadaten']):
            vals = plot_df[plot_df['Bedingung'] == cond]['VER (%)']
            ax.text(i, vals.max() + 2, f'$M={vals.mean():.1f}\\%$',
                    ha='center', va='bottom', fontsize=11)

        test = res['ver_test']
        sig_text = f"$p={test['p']:.3f}$" if test['p'] >= 0.001 else "$p<0.001$"
        sig_text += f"\n$d={test['d']:.2f}$"
        y_max = plot_df['VER (%)'].max() + 5
        ax.plot([0, 0, 1, 1], [y_max, y_max + 1, y_max + 1, y_max], 'k-', linewidth=1)
        ax.text(0.5, y_max + 1.5, sig_text, ha='center', va='bottom', fontsize=10)

        ax.set_title(config_name, fontsize=14, fontweight='bold')
        ax.set_xlabel('')
        if idx == 0:
            ax.set_ylabel('VER Success Rate (%)')
        ax.set_ylim(-5, 120)

    fig.suptitle('VER: Mit vs. Ohne Metadaten', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    _save_fig(fig, '05-rq4-ver-boxplot')

### Verteilung der ΔSEVQ-Werte

Histogramm der paarweisen Differenzen ΔSEVQ = SEVQ(mit Meta) − SEVQ(ohne Meta). Die rote gestrichelte Linie markiert die Null (kein Effekt), die dunkle Linie den beobachteten Mittelwert. Der Kasten oben rechts fasst die MBR und die Anzahl der besseren / gleichen / schlechteren Datensätze zusammen.


In [ ]:
def plot_delta_sevq(all_results):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    for idx, (config_name, res) in enumerate(all_results.items()):
        ax = axes[idx]
        delta = res['sevq_paired']['delta_sevq']

        sns.histplot(delta, bins=15, kde=True, color='#009B91', alpha=0.6, ax=ax,
                     edgecolor='white', linewidth=0.5)
        ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Δ = 0')
        ax.axvline(x=delta.mean(), color='#334152', linestyle='-', linewidth=2,
                   label=f'$M = {delta.mean():.2f}$')

        ds = res['delta_stats']
        text = (f"MBR = {ds['mbr']:.0f}%\n"
                f"$n_{{+}} = {ds['n_improved']}$, "
                f"$n_{{-}} = {ds['n_worse']}$, "
                f"$n_{{=}} = {ds['n_equal']}$")
        ax.text(0.98, 0.95, text, transform=ax.transAxes, ha='right', va='top',
                fontsize=11, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

        ax.set_title(config_name, fontsize=14, fontweight='bold')
        ax.set_xlabel('ΔSEVQ (Meta − Ohne Meta)')
        if idx == 0:
            ax.set_ylabel('Häufigkeit')
        ax.legend(fontsize=10, loc='upper left')

    fig.suptitle('Verteilung der SEVQ-Differenzen (Mit − Ohne Metadaten)',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    _save_fig(fig, '05-rq4-delta-sevq')

### Heatmap der SEVQ-Dimensionen

Die sechs SEVQ-Dimensionen nebeneinander für beide Konfigurationen, jeweils mit und ohne Metadaten. Eingefärbt nach Score-Höhe (`RdYlGn`, 5–10). Hilfreich, um zu sehen, *welche* Dimension von Metadaten profitiert und welche nicht.


In [ ]:
def plot_sevq_dimensions_heatmap(all_results):
    rows = []
    for config_name, res in all_results.items():
        dims_paired = res['sevq_dims_paired']
        for dim in METRICS:
            meta_mean = dims_paired[f'{dim}_meta'].mean()
            no_meta_mean = dims_paired[f'{dim}_no_meta'].mean()
            rows.append({'Konfiguration': f'{config_name}\nMit Meta', 'Dimension': dim, 'Score': meta_mean})
            rows.append({'Konfiguration': f'{config_name}\nOhne Meta', 'Dimension': dim, 'Score': no_meta_mean})

    heatmap_df = pd.DataFrame(rows)
    pivot = heatmap_df.pivot(index='Konfiguration', columns='Dimension', values='Score')
    pivot = pivot[METRICS]
    row_order = ['DE+Python\nMit Meta', 'DE+Python\nOhne Meta',
                 'EN+Python\nMit Meta', 'EN+Python\nOhne Meta']
    pivot = pivot.reindex(row_order)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=5, vmax=10,
                ax=ax, linewidths=1, linecolor='white', cbar_kws={'label': 'SEVQ Score'},
                annot_kws={'fontsize': 12, 'fontweight': 'bold'})
    ax.set_title('SEVQ-Dimensionen: Mit vs. Ohne Metadaten', fontsize=14, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    plt.tight_layout()
    _save_fig(fig, '05-rq4-dimensions-heatmap')

### A/B-Präferenzen aus der Studie

Diverging-Bar-Chart der Präferenzen aus den A/B-Fragen. Links: Anteil der Stimmen für „ohne Metadaten“, rechts: Anteil für „mit Metadaten“. Die Sterne neben dem Balken markieren das Signifikanzniveau des Binomialtests (`***` < .001, `**` < .01, `*` < .05, sonst `n.s.`).


In [ ]:
def plot_ab_preferences(ab_results):
    fig, ax = plt.subplots(figsize=(10, 5))
    configs = ['DE+Python', 'EN+Python', 'Gesamt']

    for i, config in enumerate(configs):
        res = ab_results[config]
        prop_meta = res['prop_meta'] * 100
        prop_no_meta = (1 - res['prop_meta']) * 100

        ax.barh(i, prop_meta, color=PALETTE_META['Mit Metadaten'], height=0.6,
                edgecolor='white', linewidth=1)
        ax.barh(i, -prop_no_meta, color=PALETTE_META['Ohne Metadaten'], height=0.6,
                edgecolor='white', linewidth=1)

        ax.text(prop_meta + 1, i, f'{res["n_meta"]} ({prop_meta:.1f}%)',
                va='center', ha='left', fontsize=11, fontweight='bold')
        ax.text(-prop_no_meta - 1, i, f'{res["n_no_meta"]} ({prop_no_meta:.1f}%)',
                va='center', ha='right', fontsize=11, fontweight='bold')

        sig = '***' if res['p'] < 0.001 else '**' if res['p'] < 0.01 else '*' if res['p'] < 0.05 else 'n.s.'
        ax.text(55, i, sig, va='center', ha='left', fontsize=10, color='gray')

    ax.set_yticks(range(len(configs)))
    ax.set_yticklabels(configs, fontsize=12)
    ax.axvline(x=0, color='black', linewidth=1)
    ax.set_xlabel('Präferenz (%)')
    ax.set_xlim(-80, 80)
    ax.set_title('A/B-Präferenzen: Mit vs. Ohne Metadaten', fontsize=14, fontweight='bold')

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=PALETTE_META['Mit Metadaten'], label='Mit Metadaten'),
        Patch(facecolor=PALETTE_META['Ohne Metadaten'], label='Ohne Metadaten')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    ax.axvline(x=50, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.axvline(x=-50, color='gray', linestyle=':', linewidth=1, alpha=0.5)

    plt.tight_layout()
    _save_fig(fig, '05-rq4-ab-preferences')

### Zusammenfassende Übersicht

Drei-Panel-Plot, der die wichtigsten Kennzahlen in einer Abbildung bündelt: SEVQ-Total, VER und A/B-Präferenz. Die Fehlerbalken sind 95%-Konfidenzintervalle. Diese Abbildung ist die Hauptgrafik im RQ-4-Kapitel.


In [ ]:
def plot_summary(all_results, ab_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    configs = list(all_results.keys())
    x = np.arange(len(configs))
    width = 0.35

    # (a) SEVQ Total
    ax = axes[0]
    for i, config in enumerate(configs):
        paired = all_results[config]['sevq_paired']
        for j, (col, label) in enumerate([('sevq_total_meta', 'Mit Metadaten'),
                                           ('sevq_total_no_meta', 'Ohne Metadaten')]):
            vals = paired[col]
            m = vals.mean()
            lo, hi = ci95(vals)
            ax.bar(i + (j - 0.5) * width, m, width, yerr=m - lo, capsize=5,
                   color=PALETTE_META[label], label=label if i == 0 else '')
    ax.set_ylabel('SEVQ Total (0–60)')
    ax.set_title('(a) SEVQ-Total', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.set_ylim(35, 58)
    ax.legend(fontsize=9)

    # (b) VER
    ax = axes[1]
    for i, config in enumerate(configs):
        paired = all_results[config]['ver_paired']
        for j, (col, label) in enumerate([('ver_success_rate_meta', 'Mit Metadaten'),
                                           ('ver_success_rate_no_meta', 'Ohne Metadaten')]):
            vals = paired[col]
            m = vals.mean()
            lo, hi = ci95(vals)
            ax.bar(i + (j - 0.5) * width, m, width, yerr=m - lo, capsize=5,
                   color=PALETTE_META[label])
    ax.set_ylabel('VER Success Rate (%)')
    ax.set_title('(b) VER', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.set_ylim(0, 100)

    # (c) A/B Preferences
    ax = axes[2]
    ab_configs = ['DE+Python', 'EN+Python', 'Gesamt']
    x_ab = np.arange(len(ab_configs))
    meta_props = [ab_results[c]['prop_meta'] * 100 for c in ab_configs]
    no_meta_props = [(1 - ab_results[c]['prop_meta']) * 100 for c in ab_configs]
    meta_ci_lower = [ab_results[c]['ci_lower'] * 100 for c in ab_configs]
    meta_ci_upper = [ab_results[c]['ci_upper'] * 100 for c in ab_configs]
    meta_errs = [[m - l for m, l in zip(meta_props, meta_ci_lower)],
                 [u - m for m, u in zip(meta_props, meta_ci_upper)]]

    ax.bar(x_ab - width / 2, meta_props, width, yerr=meta_errs, capsize=5,
           color=PALETTE_META['Mit Metadaten'])
    ax.bar(x_ab + width / 2, no_meta_props, width, capsize=5,
           color=PALETTE_META['Ohne Metadaten'])
    ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_ylabel('Präferenz (%)')
    ax.set_title('(c) A/B-Präferenz', fontsize=13, fontweight='bold')
    ax.set_xticks(x_ab)
    ax.set_xticklabels(ab_configs)
    ax.set_ylim(0, 100)

    fig.suptitle('RQ-4: Zusammenfassender Vergleich', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    _save_fig(fig, '05-rq4-summary')

## Export der Kennzahlen als JSON

`export_summary` schreibt alle Kennzahlen in eine `rq4_summary.json`. Dort landen pro Konfiguration die deskriptiven Werte (M, SD, Median), die Testergebnisse (t, p, d, KI), die ΔSEVQ-Statistiken inkl. MBR und die Werte der einzelnen SEVQ-Dimensionen — sowie die A/B-Ergebnisse aller drei Vergleichsblöcke.

Diese Datei wird direkt vom Typst-Quelltext der Thesis konsumiert, sodass die Zahlen im Text immer mit denen im Notebook übereinstimmen. Der `default`-Lambda-Konverter sorgt dafür, dass NumPy-Typen sauber als Python-Typen serialisiert werden.


In [ ]:
def export_summary(all_results, ab_results):
    summary = {}
    for config_name, res in all_results.items():
        paired = res['sevq_paired']
        ver_p = res['ver_paired']
        meta_vals = paired['sevq_total_meta']
        no_meta_vals = paired['sevq_total_no_meta']
        ver_meta_vals = ver_p['ver_success_rate_meta']
        ver_no_meta_vals = ver_p['ver_success_rate_no_meta']

        summary[config_name] = {
            'n_paired_sevq': len(paired),
            'n_paired_ver': len(ver_p),
            'sevq_meta_mean': round(float(meta_vals.mean()), 2),
            'sevq_meta_sd': round(float(meta_vals.std()), 2),
            'sevq_meta_median': round(float(meta_vals.median()), 2),
            'sevq_no_meta_mean': round(float(no_meta_vals.mean()), 2),
            'sevq_no_meta_sd': round(float(no_meta_vals.std()), 2),
            'sevq_no_meta_median': round(float(no_meta_vals.median()), 2),
            'sevq_t': round(res['sevq_test']['t'], 3),
            'sevq_p': round(res['sevq_test']['p'], 4),
            'sevq_d': round(res['sevq_test']['d'], 3),
            'sevq_ci_lower': round(res['sevq_test']['ci_lower'], 2),
            'sevq_ci_upper': round(res['sevq_test']['ci_upper'], 2),
            'ver_meta_mean': round(float(ver_meta_vals.mean()), 1),
            'ver_meta_sd': round(float(ver_meta_vals.std()), 1),
            'ver_no_meta_mean': round(float(ver_no_meta_vals.mean()), 1),
            'ver_no_meta_sd': round(float(ver_no_meta_vals.std()), 1),
            'ver_t': round(res['ver_test']['t'], 3),
            'ver_p': round(res['ver_test']['p'], 4),
            'ver_d': round(res['ver_test']['d'], 3),
            'delta_sevq_mean': round(res['delta_stats']['mean'], 2),
            'delta_sevq_sd': round(res['delta_stats']['sd'], 2),
            'mbr_sevq': round(res['delta_stats']['mbr'], 1),
            'n_improved': res['delta_stats']['n_improved'],
            'n_equal': res['delta_stats']['n_equal'],
            'n_worse': res['delta_stats']['n_worse'],
            'delta_ver_mean': round(res['delta_ver_stats']['mean'], 1),
            'mbr_ver': round(res['delta_ver_stats']['mbr'], 1),
            'dimensions': {
                dim: {
                    'meta_mean': round(dr['meta_mean'], 2),
                    'no_meta_mean': round(dr['no_meta_mean'], 2),
                    'diff': round(dr['diff_mean'], 2),
                    'p': round(dr['p'], 4),
                    'd': round(dr['d'], 3)
                }
                for dim, dr in res['dim_results'].items()
            }
        }

    summary['ab_results'] = {
        config_name: {
            'n_meta': r['n_meta'],
            'n_no_meta': r['n_no_meta'],
            'n_total': r['n_total'],
            'prop_meta': round(r['prop_meta'] * 100, 1),
            'p': round(r['p'], 4)
        }
        for config_name, r in ab_results.items()
    }

    out_path = OUTPUT_DIR / 'rq4_summary.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False,
                  default=lambda o: int(o) if isinstance(o, np.integer) else
                                    float(o) if isinstance(o, np.floating) else
                                    o.tolist() if isinstance(o, np.ndarray) else None)
    print(f"  -> {out_path}")

## Auswertung ausführen

Letzte Zelle: `generate_plots` bündelt alle Plot-Aufrufe in einer Funktion (damit `run_analysis` sie geschlossen aufrufen kann), danach wird `run_analysis()` gestartet. Die komplette Pipeline läuft durch:

1. beide Konfigurationen einzeln auswerten,
2. global aggregieren,
3. A/B-Studie auswerten,
4. Plots schreiben,
5. JSON-Zusammenfassung exportieren.

Die Konsolenausgabe unten ist das vollständige Log des letzten Durchlaufs und dient als Reproduktionsnachweis.


In [ ]:
def generate_plots(all_results, ab_results):
    plot_sevq_boxplot(all_results)
    plot_ver_boxplot(all_results)
    plot_delta_sevq(all_results)
    plot_sevq_dimensions_heatmap(all_results)
    plot_ab_preferences(ab_results)
    plot_summary(all_results, ab_results)

run_analysis()